# Computer Vision Pipeline 

### Renaming images

In [1]:
import os

# 🔹 CHANGE THIS to your dataset root
DATASET_ROOT = "/Users/arnavkarnik/Desktop/AIBD_Project/Aishwarya thesis"

# Supported image extensions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def rename_images_sequentially(root_dir):
    for class_name in sorted(os.listdir(root_dir)):
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        # Collect valid image files only
        images = [
            f for f in os.listdir(class_path)
            if os.path.splitext(f)[1].lower() in IMAGE_EXTS
        ]

        images.sort()  # deterministic order

        print(f"\nRenaming class: {class_name} | Images: {len(images)}")

        for idx, filename in enumerate(images, start=1):
            ext = os.path.splitext(filename)[1].lower()
            new_name = f"{class_name}_{idx:04d}{ext}"

            old_path = os.path.join(class_path, filename)
            new_path = os.path.join(class_path, new_name)

            # Safety check
            if old_path != new_path:
                os.rename(old_path, new_path)

        print(f"✔ Done: {class_name}")

if __name__ == "__main__":
    rename_images_sequentially(DATASET_ROOT)



Renaming class: BP | Images: 2942
✔ Done: BP

Renaming class: HSP | Images: 1052
✔ Done: HSP

Renaming class: NEGATIVES | Images: 817
✔ Done: NEGATIVES

Renaming class: PF | Images: 164
✔ Done: PF

Renaming class: PV | Images: 2720
✔ Done: PV

Renaming class: VASCULITIS | Images: 436
✔ Done: VASCULITIS

Renaming class: eba | Images: 198
✔ Done: eba


In [2]:
import os
import cv2
import numpy as np

from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif


# =========================
# CONFIGURATION
# =========================

DATASET_ROOT = "/Users/arnavkarnik/Desktop/AIBD_Project/Aishwarya thesis"     # PEMPHIGUS / LINEAR_BMZ / VASCULITIC / NEGATIVE
OUTPUT_ROOT  = "/Users/arnavkarnik/Desktop/AIBD_Project/preprocessed_images"       # saved images + masks
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

CLAHE_CLIP   = 1.8
GAUSS_SIGMA  = 0.7
SELECT_K     = 40


# =========================
# PREPROCESSING FUNCTIONS
# =========================

def preprocess_image(img):
    green = img[:, :, 1]

    green = cv2.normalize(green, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8))
    green = clahe.apply(green)

    green = cv2.GaussianBlur(green, (3, 3), GAUSS_SIGMA)

    return green


def foreground_mask(img):
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    masked = cv2.bitwise_and(img, img, mask=mask)
    return masked, mask


# =========================
# FEATURE EXTRACTION
# =========================

def intensity_features(img):
    px = img[img > 0]
    if len(px) == 0:
        return [0]*7

    return [
        np.mean(px),
        np.std(px),
        skew(px),
        kurtosis(px),
        np.percentile(px, 10),
        np.percentile(px, 50),
        np.percentile(px, 90)
    ]


def texture_features(img):
    img_q = (img / 8).astype(np.uint8)

    glcm = graycomatrix(
        img_q,
        distances=[1, 2, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        symmetric=True,
        normed=True
    )

    glcm_feats = []
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        glcm_feats.extend(graycoprops(glcm, prop).flatten())

    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)

    return glcm_feats + lbp_hist.tolist()


def shape_features(img):
    edges = cv2.Canny(img, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return [0]*11

    cnt = max(contours, key=cv2.contourArea)

    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)

    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = w / h if h > 0 else 0
    circularity = (4 * np.pi * area) / (perimeter**2 + 1e-6)

    hu = cv2.HuMoments(cv2.moments(cnt)).flatten()

    return [area, perimeter, aspect_ratio, circularity] + hu.tolist()


def topology_features(mask):
    binary = (mask > 0).astype(np.uint8)

    skeleton = skeletonize(binary).astype(np.uint8)
    skeleton_length = np.sum(skeleton)

    regions = regionprops(label(binary))
    num_components = len(regions)
    mean_area = np.mean([r.area for r in regions]) if regions else 0

    return [skeleton_length, num_components, mean_area]


def extract_features(img):
    pre = preprocess_image(img)
    masked, mask = foreground_mask(pre)

    features = []
    features.extend(intensity_features(masked))
    features.extend(texture_features(masked))
    features.extend(shape_features(masked))
    features.extend(topology_features(mask))

    return features, masked, mask


# =========================
# MAIN PIPELINE
# =========================

X = []
y = []

for label_name in sorted(os.listdir(DATASET_ROOT)):
    class_dir = os.path.join(DATASET_ROOT, label_name)
    if not os.path.isdir(class_dir):
        continue

    out_img_dir  = os.path.join(OUTPUT_ROOT, label_name, "images")
    out_mask_dir = os.path.join(OUTPUT_ROOT, label_name, "masks")
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    idx = 0

    for fname in sorted(os.listdir(class_dir)):
        if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
            continue

        img = cv2.imread(os.path.join(class_dir, fname))
        if img is None:
            continue

        feats, final_img, mask = extract_features(img)

        cv2.imwrite(os.path.join(out_img_dir, f"{label_name}_{idx:04d}.png"), final_img)
        cv2.imwrite(os.path.join(out_mask_dir, f"{label_name}_{idx:04d}.png"), mask)

        X.append(feats)
        y.append(label_name)

        idx += 1


# =========================
# FEATURE SCALING & SELECTION
# =========================

X = np.array(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=SELECT_K)
X_selected = selector.fit_transform(X_scaled, y)

print("Final feature matrix shape:", X_selected.shape)
print("Labels:", set(y))


Final feature matrix shape: (8329, 40)
Labels: {'PF', 'eba', 'PV', 'HSP', 'BP', 'VASCULITIS', 'NEGATIVES'}
